# Bedrock (mocked) + the capstone deploy map

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 33 §33.6, §33.9 · type: walkthrough* 🔧 *(carries the chapter's Build: “deploy the capstone on AWS” — as a deploy map + notes)*

*One-line promise:* make a **Bedrock-shaped** model call behind a mock, then read the capstone's full AWS deployment as a **component→service map** you can hand straight to the Ch 36 Terraform — all **offline, no Bedrock access, no spend**.

## 🧠 Why this matters

**Amazon Bedrock** is AWS's managed gateway to foundation models — Anthropic's Claude and others — behind **one API, inside your AWS account and security perimeter**. For a team already on AWS it's often the shortest path to production model access, with requests and responses staying within your AWS boundary (a real privacy/compliance win). The model-call *shape* is what you need in your hands: a `bedrock-runtime` `invoke_model` call carrying an **Anthropic Claude** request/response body. Once you can read that envelope, the rest of this notebook zooms out to the payoff of the whole chapter: a single map that places **every** capstone component onto a concrete AWS service — the exact target the Ch 36 Terraform must build.

## Objectives & prereqs

**By the end you can:**
- Issue a `bedrock-runtime` **`invoke_model`** call in the **Anthropic Claude** request shape and read the response **envelope** (know exactly where the text lives).
- Explain Bedrock's role: the managed **model gateway** inside your perimeter — and where its managed pieces (Knowledge Bases, Guardrails) map onto patterns you built by hand.
- Render the **§33.9 component→service deploy map** for the capstone.
- Walk the **§33.9 production-readiness checklist** — the bar Ch 36's Terraform must meet.

**Prereqs:** [`33-01`](./33-01-iam-and-least-privilege.ipynb) and [`33-02`](./33-02-s3-dynamodb-sqs-on-moto.ipynb); Ch 11 (model APIs); Ch 13/41 for the Knowledge Bases ↔ RAG and Guardrails ↔ safety ties.

**Packages:** standard library only for the default path. The optional **live** path imports `boto3` (chapter extra: `pip install boto3`) and needs Bedrock model access — never required to complete the notebook.

**Cost:** **mockable.** `MOCK=1` (default) serves a **canned, realistic** Claude completion from `data/` — **no spend, no Bedrock access**. ⚠️ `MOCK=0` makes a **real AWS Bedrock** call (real per-token charges; needs Bedrock model access + credentials) — opt-in only; budget a couple of short Claude completions. The deploy-map and checklist cells are **pure-offline** regardless.

## Setup

In [ ]:
import json
import os
import random
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()  # reads a git-ignored .env if present; never hardcode keys or credentials

# MOCK=1 (default): serve a canned, realistic Claude-via-Bedrock response from data/ ->
# FREE, OFFLINE, no Bedrock access. MOCK=0: a REAL bedrock-runtime invoke_model call
# (real tokens, real $$, needs model access + credentials). Default stays mocked on purpose.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(33)  # determinism for anything stochastic on our side

REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
DATA = Path("data")

# Bedrock addresses Anthropic's Claude by a model ID (the on-demand inference profile id in
# real accounts). The request/response BODY is the standard Anthropic Messages shape.
MODEL_ID = "anthropic.claude-opus-4-8"

if MOCK:
    print("MOCK mode: canned Bedrock/Claude response from data/. No Bedrock access, nothing billed.")
else:
    if not (os.getenv("AWS_ACCESS_KEY_ID") and os.getenv("AWS_SECRET_ACCESS_KEY")):
        raise RuntimeError(
            "COMPANION_MOCK=0 but AWS credentials are not set. Provide AWS creds with Bedrock "
            "model access in your environment/.env, or use COMPANION_MOCK=1 (offline)."
        )
    print(f"LIVE mode: real bedrock-runtime invoke_model on {MODEL_ID} in {REGION}. Tokens WILL be billed.")

## 1 · Bedrock as the managed model gateway (§33.6)

Mental model: Bedrock is **one API in front of many models**, living *inside* your AWS account. You don't run any inference infrastructure; you call `bedrock-runtime`'s `invoke_model` with a **model ID** and a **body**, and for Claude that body is the ordinary **Anthropic Messages** request — `anthropic_version`, `max_tokens`, `messages`. The response body comes back in the Anthropic shape too. Because it's all within your account boundary, your prompts and completions don't leave your perimeter — the compliance advantage that makes Bedrock attractive on AWS.

Here's the request body we'll send (identical for mock and live — only the *transport* differs):

In [ ]:
# The Anthropic Messages request body — exactly what you'd send to Claude directly,
# just delivered through Bedrock's invoke_model(body=...).
request_body = {
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 256,
    "messages": [
        {"role": "user", "content": "In two sentences, what is Amazon Bedrock and when should I use it?"}
    ],
}
print(json.dumps(request_body, indent=2))

### 🔮 Predict: where does the text live?

We're about to invoke the model and parse the result. The response is the **Anthropic Messages** envelope. **Predict** the path to the model's actual text — is it…

- (a) `response["text"]`,
- (b) `response["choices"][0]["message"]["content"]` (that's the *OpenAI* shape), or
- (c) `response["content"][0]["text"]` with usage under `response["usage"]`?

Write your guess, then run the next two cells.

In [ ]:
def invoke_bedrock(body: dict) -> dict:
    """Return the parsed Anthropic-shaped response body.

    MOCK=1: load a canned, realistic response from data/ (no Bedrock access, no spend).
    MOCK=0: a REAL bedrock-runtime invoke_model call. The *parsing* below is identical for
    both, because the response body shape is the same — that's the whole point of rehearsing
    the envelope offline.
    """
    if MOCK:
        return json.loads((DATA / "bedrock-canned-response.json").read_text(encoding="utf-8"))

    # --- live path (opt-in; real tokens billed) ---
    import boto3  # chapter extra: pip install boto3

    client = boto3.client("bedrock-runtime", region_name=REGION)
    resp = client.invoke_model(
        modelId=MODEL_ID,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json",
    )
    return json.loads(resp["body"].read())


response = invoke_bedrock(request_body)
print(json.dumps(response, indent=2)[:700])

In [ ]:
# The answer to the prediction: Anthropic puts text in content[].text and tokens in usage.
text = response["content"][0]["text"]
usage = response["usage"]

print("MODEL TEXT:\n", text)
print("\nstop_reason:", response["stop_reason"])
print("tokens in/out:", usage["input_tokens"], "/", usage["output_tokens"])

**What you just saw.** The answer is **(c)**: in the Anthropic shape the text is at `content[0]["text"]` (a list of content blocks), with token counts under `usage`. That's the OpenAI `choices[0].message.content` shape's counterpart — getting them confused is the #1 Bedrock-with-Claude parsing bug. Because the **body** is plain Anthropic Messages, code you wrote for the Claude API in Ch 11 ports to Bedrock with little more than a transport swap.

## 2 · Bedrock's managed pieces ↔ patterns you already built

Bedrock bundles higher-level features that map **directly** onto things you built by hand earlier in the book. Seeing the correspondence keeps you from treating them as magic — and helps you decide, deliberately, which layers to outsource. This is a **comparison table, not new code.**

In [ ]:
# A plain mapping table — pure-offline, no API.
mappings = [
    ("Knowledge Bases", "Ch 13 RAG",
     "Managed chunk/embed/store/retrieve over S3 -> a retriever without building ingestion."),
    ("Guardrails", "Ch 41 safety + Ch 13 grounding",
     "Content filters, denied topics, PII redaction, and a contextual-grounding check."),
    ("Agents", "Parts IV-V orchestration",
     "A managed agent runtime; convenient, but less control than your own loop."),
    ("Provisioned throughput", "Ch 39 capacity",
     "Reserved model capacity for steady load vs. on-demand."),
    ("Cross-region inference", "Ch 39 availability",
     "Spread requests across regions to absorb spikes and raise availability."),
]

print(f"{'Bedrock feature':24} {'You built it in':28} What it gives you")
print("-" * 100)
for feature, builtin, gives in mappings:
    print(f"{feature:24} {builtin:28} {gives}")

**The Guardrails detail worth knowing.** Its **contextual grounding check** scores whether a response is actually grounded in the retrieved source and relevant to the query, then blocks or flags answers below your threshold — a *managed hallucination guardrail* that is the Ch 13 grounding discipline and Ch 41 safety layer, delivered as a policy you attach to model calls.

## 🎯 Senior lens: outsource the model plane, keep the orchestration

Bedrock's managed Agents and Knowledge Bases get you to a demo fast — and can box you into AWS's abstractions and pricing, away from the framework flexibility you built in Parts IV–V. The §33.6 senior pattern: use Bedrock for **model access** (the inference API, with its security and compliance benefits) while keeping your **orchestration in your own code** behind your own gateway (Ch 39). You keep portability and control where they matter and let AWS manage the model plumbing. The decision isn't all-or-nothing — it's choosing, layer by layer, what to outsource. Our capstone takes exactly this stance: Bedrock (or a direct provider) for inference; the agent loop, tools, memory, and evals stay ours.

## 3 · 🔧 Build: the capstone deploy map (§33.9)

Now the chapter's Build, as the deliverable it really is: a **component→service map**. The shape is the thin-API, background-worker pattern from Part VII, deployed across AZs for resilience. Every component you built lands on a concrete AWS service — and this table *is* the spec the Ch 36 Terraform implements.

In [ ]:
# The §33.9 component -> AWS service map. Pure data; render it as a table.
deploy_map = [
    ("FastAPI service",      "ECS on Fargate (private subnets)", "Thin API; containers, no servers to patch"),
    ("Celery workers",       "ECS on Fargate (private subnets)", "Background work; auto-scaled; spot for batch"),
    ("Public entry point",   "CloudFront -> ALB",                "CDN edge in front of the load balancer"),
    ("Task queue (API->worker)", "SQS",                          "At-least-once handoff (notebook 33-02)"),
    ("Relational + vectors", "RDS Postgres + pgvector",          "Primary store; multi-AZ; KMS at rest"),
    ("Cache / broker",       "ElastiCache (Redis)",             "Caching and Celery broker"),
    ("Objects / artifacts",  "S3",                               "Documents, exports, model artifacts"),
    ("Model inference",      "Bedrock",                          "Managed model gateway in-perimeter"),
    ("Logs / metrics / traces", "CloudWatch (+ X-Ray)",         "AWS-native observability (Part VI)"),
    ("Identity per service", "IAM roles",                        "Least-privilege role, no long-lived keys"),
    ("Network isolation",    "VPC (public + private subnets)",  "Only the ALB is public; data is private"),
]

print(f"{'Capstone component':26} {'AWS service':32} Why")
print("-" * 104)
for component, service, why in deploy_map:
    print(f"{component:26} {service:32} {why}")

**What you just saw.** The backend from Part VII, placed onto real infrastructure: API and workers as **Fargate** containers in **private** subnets; an **ALB** behind **CloudFront** as the only public door; **SQS** carrying tasks; **RDS+pgvector / ElastiCache / S3** as the data layer; **Bedrock** for models; **CloudWatch** watching it all; every component assuming its own tightly-scoped **IAM role**, inside a **VPC**. You won't click this together — the next chapters define it as code — but you can now read the target.

## 4 · 📋 Production-readiness checklist (§33.9)

This is the bar Ch 36's Terraform must meet — the same checklist from the chapter, so you can audit the IaC against it line by line.

In [ ]:
checklist = [
    ("IAM",          "Roles (not keys) per service; least-privilege; no *:*"),
    ("Network",      "Private subnets for app/data; only the LB public; tight security groups"),
    ("Secrets",      "Secrets Manager / Parameter Store, fetched via role; never in env/code/prompts"),
    ("Compute",      "Fargate for API + workers; right-sized, auto-scaled; spot for batch"),
    ("Images",       "Built and pushed to ECR; no secrets baked in"),
    ("Data",         "RDS Postgres + pgvector; multi-AZ; tested backups; KMS at rest"),
    ("Edge",         "WAF + Shield on the CloudFront/ALB edge; Route 53 DNS + health checks"),
    ("Resilience",   "Spread across >= 2 AZs; health checks and graceful shutdown"),
    ("Cost",         "Billing alarms and tags from day one; stop non-prod off-hours"),
    ("Audit",        "CloudTrail across the account; the SOC 2 evidence trail (Ch 41)"),
    ("Observability","CloudWatch logs/metrics/alarms; tracing wired up (Part VI)"),
]

for area, requirement in checklist:
    print(f"[ ] {area:14} {requirement}")

### ⚠️ Pitfall: the mock hides a real meter

Everything above ran free because `MOCK=1` served a canned completion. The instant you flip `COMPANION_MOCK=0`, that same `invoke_model` call bills **real Bedrock tokens** on your account — and a chatty agent loop multiplies it fast. Two habits keep this safe: keep the **mock as the default** for development and CI (deterministic, $0), and when you do go live, set up the **billing alarms and tags from day one** that the checklist demands — *before* the first real call, not after the first surprise invoice.

## Recap

- **Bedrock** is a managed model gateway **inside your AWS perimeter**: `bedrock-runtime` `invoke_model` with a model ID and an **Anthropic Messages** body.
- The Claude response text lives at **`content[0].text`**, tokens under **`usage`** — not the OpenAI `choices[...]` shape.
- Bedrock's managed pieces map onto things you built by hand: **Knowledge Bases ↔ Ch 13 RAG**, **Guardrails ↔ Ch 41 safety + Ch 13 grounding**.
- Senior pattern: outsource the **model plane** to Bedrock; keep **orchestration** in your own code for portability and control.
- The **§33.9 deploy map** places every capstone component on a concrete service; the **§33.9 checklist** is the bar the Ch 36 Terraform must clear.
- The mock is free; ⚠️ live Bedrock bills real tokens — keep the mock default and wire billing alarms before going live.

## Exercises

Use the empty cells below. Each one *changes* something — predict the result first. (Solutions land in `solutions/` in Phase 2.)

1. **Read a different envelope.** Edit `data/bedrock-canned-response.json` to add a second content block (another `{"type":"text", ...}`). 🔮 Predict what `response["content"][0]["text"]` returns now, and write the loop that concatenates *all* text blocks.
2. **Map a new component.** The capstone adds a nightly **evaluation job** (Part VI). Add a row to `deploy_map` placing it on the right service(s) — and justify the choice in the “Why” column (hint: §33 AI-specific infra — Athena/Glue, or a scheduled Fargate task).
3. **Audit the checklist.** Pick three checklist items and, for each, name the **specific AWS resource** in the deploy map it constrains (e.g. *Secrets* → the Fargate task role + Secrets Manager). Which item, if skipped, fails a SOC 2 audit first?
4. **(Live, optional)** With `COMPANION_MOCK=0`, real AWS credentials, and Bedrock **model access** for `MODEL_ID`, run the `invoke_bedrock` cell. 🔮 Predict how the live `usage` token counts compare to the canned ones, and note the one line of code that changes between mock and live (transport, not parsing).

In [ ]:
# Exercise 1 — add a 2nd content block to the fixture; concatenate all text blocks.

In [ ]:
# Exercise 2 — add the nightly eval job to deploy_map with a justified service choice.

In [ ]:
# Exercise 3 — map three checklist items to specific deploy-map resources.

## Next

- 🔧 **The production version of this map:** the Ch 36 Terraform ([`templates/terraform-module/`](../../templates/terraform-module/)) and the containers from Ch 35 ([`templates/dockerfile-and-compose/`](../../templates/dockerfile-and-compose/)) turn this component→service map into real, applied infrastructure.
- 🎓 **Capstone:** this notebook produces the **AWS deploy notes / service map** consumed by [`capstone-project/infra/`](../../capstone-project/); the §33.9 architecture is the target `capstone-project/app/` and `capstone-project/workers/` deploy onto. Advances checkpoint `checkpoints/ch33-aws-deploy-map`.
- 📖 **Book:** §33.6 (Amazon Bedrock) and §33.9 (Build: deploy the capstone on AWS) — the figure, map, and checklist this notebook makes runnable.